# Using the DynamicCallable Module in baseobjects

## Introduction

The DynamicCallable module provides abstract callable classes with multiplexed binding and callback. It builds on BaseCallable and introduces MethodMultiplexer-powered selection of both how the object binds (descriptor behavior) and how it calls.

This tutorial will guide you through:
- What DynamicCallable is and how it differs from BaseCallable
- How binding and calling are delegated to MethodMultiplexer
- How to switch bind_method and call_method at runtime
- Interactions with DynamicMethod and DynamicFunction

**Prerequisites:**
- Familiarity with BaseCallable
- Understanding of descriptor protocol and callables

### Table of Contents
- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting--FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.functions import DynamicCallable, MethodMultiplexer
from baseobjects.bases.basecallable import BaseCallable

## Core Functionality

DynamicCallable is an abstract class that:
- Exposes bind_multiplexer and call_multiplexer (both MethodMultiplexer)
- Supports selecting bind_method and call_method via properties or in construct()
- Delegates __get__ to bind_multiplexer and __call__ to call_multiplexer

We'll implement a minimal subclass to explore binding and calling multiplexers.

In [2]:
class DemoDynamicCallable(DynamicCallable):
    """Demo subclass to showcase multiplexed bind and call."""

    def call_wrapped(self, *args, **kwargs):
        # default call path when a wrapped function exists; for demo we'll just echo
        return ("call_wrapped", args, kwargs)

    # Additional call strategies for the multiplexer to select
    def call_with_logging(self, *args, **kwargs):
        print("[log] calling with:", args, kwargs)
        return ("call_with_logging", args, kwargs)

    # Binding strategies
    def bind_builtin(self, instance=None, owner=None):
        # default binding behavior mimicking BaseCallable.bind_builtin
        print("bind builtin with:", instance, owner)
        return super().bind_builtin(instance=instance, owner=owner)

    def bind_self(self, instance=None, owner=None):
        # provide an alternate binding path returning a bound method to instance
        print("bind self with:", instance, owner)
        return self

# Create the object with no wrapped function for demonstration
obj = DemoDynamicCallable()

# Inspect current methods selected by multiplexers
print("Initial bind method:", obj.bind_method)
print("Initial call method:", obj.call_method)

# Switch call method to our custom strategy
obj.call_method = "call_with_logging"
print("Switched call method:", obj.call_method)
print("Call result:", obj(1, x=2))

Initial bind method: bind_builtin
Initial call method: call_wrapped
Switched call method: call_with_logging
[log] calling with: (1,) {'x': 2}
Call result: ('call_with_logging', (1,), {'x': 2})


### Method Multiplexer Highlight: Binding vs Callback

- bind_multiplexer controls how the object binds when accessed as a descriptor (via __get__).
- call_multiplexer controls how the object handles calls (via __call__).

You can select methods using:
- Properties: obj.bind_method = "...", obj.call_method = "..."
- During construction: DemoDynamicCallable(None, bind_method="...", call_method="...")

Let's demonstrate binding differences.

In [3]:
class Container:
    demo = DemoDynamicCallable()

c = Container()

# Default binding
bound = c.demo
print("Type of bound:", type(bound))

# Switch binding strategy and rebind
Container.demo.bind_method = "bind_self"
bound2 = c.demo
print("Type with bind_self:", type(bound2))

bind builtin with: <__main__.Container object at 0x000001DCD9A94FB0> <class '__main__.Container'>
Type of bound: <class 'method'>
bind builtin with: None <class '__main__.Container'>
bind self with: <__main__.Container object at 0x000001DCD9A94FB0> <class '__main__.Container'>
Type with bind_self: <class '__main__.DemoDynamicCallable'>


## Module Interaction

DynamicCallable interacts closely with MethodMultiplexer:
- The multiplexer maintains a registry of available methods and the selected key.
- DynamicMethod and DynamicFunction are built on DynamicCallable and inherit this multiplexed behavior.

We'll quickly show that the registry and selection are stateful and pickle-friendly.

In [4]:
import pickle

p = pickle.dumps(Container.demo)
restored = pickle.loads(p)
print("Restored bind selected:", restored.bind_method)
print("Restored call selected:", restored.call_method)

bind self with: None <class '__main__.Container'>
Restored bind selected: bind_self
Restored call selected: call_wrapped


## Advanced Features

- You can preselect strategies in construct():

In [5]:
obj2 = DemoDynamicCallable(None, bind_method="bind_builtin", call_method="call_with_logging")
print(obj2.bind_method, obj2.call_method)
print(obj2("advanced"))

bind_builtin call_with_logging
[log] calling with: ('advanced',) {}
('call_with_logging', ('advanced',), {})


## Examples

- Toggle call strategies at runtime for instrumentation (logging/metrics)
- Choose different binding behavior for special descriptor use-cases

In [6]:
obj3 = DemoDynamicCallable()
obj3.call_method = "call_wrapped"
print(obj3("A"))
obj3.call_method = "call_with_logging"
print(obj3("B"))

('call_wrapped', ('A',), {})
[log] calling with: ('B',) {}
('call_with_logging', ('B',), {})


## API Highlights

- Properties: bind_method, call_method
- Attributes: bind_multiplexer, call_multiplexer
- construct(func=None, *, bind_method=None, call_method=None, **kwargs)
- __get__ delegates to bind_multiplexer
- __call__ delegates to call_multiplexer

See also: tutorials for MethodMultiplexer.

## Troubleshooting / FAQs

### Q: My custom method name isn't being used by the multiplexer.

A: Ensure the method exists on the instance and the exact string is used when selecting via bind_method/call_method.

### Q: Pickling loses my selected strategy.

A: DynamicCallable persists the multiplexer registry and selected key in __getstate__/__setstate__.

## Conclusion and Next Steps

You learned how DynamicCallable uses MethodMultiplexer to control both binding and calling behavior. Explore DynamicMethod and DynamicFunction for specialized method/function semantics.
